# 01 — Hydrogen Bond Analysis

Hydrogen bonds between chain C and chains A/B across three replicate
trajectories, resolved per residue and in aggregate.

### What this notebook does
1. Loads each replicate (one shared PDB topology + one DCD per replicate).
2. Runs `MDAnalysis.analysis.hydrogenbonds.HydrogenBondAnalysis` in **both
   chemical directions** (C → A/B and A/B → C), for two scopes:
   - **target** — chain C against the specific residues in `TARGET_RESIDUES`
   - **total** — chain C against *all* of chains A/B (whole interface)
3. Discards the first `FRAMES_TO_SKIP` frames of each replicate as
   equilibration.
4. Reports per-residue H-bond frequency (bonds per production frame) and mean
   H-bonds per frame, per replicate and pooled.
5. Records, for every frame, the residue-level pairs and the Cα RMSD to the
   replicate average, and pickles them for
   `02_representative_frame_analysis.ipynb`.

### Geometric criteria
Maximum donor–acceptor distance **3.5 Å**, minimum donor–hydrogen–acceptor
angle **150°**.

The first **100 ns** of each replicate is discarded as equilibration. With
coordinates saved every 100 ps, that is 1,000 of the 10,000 frames per
replicate, leaving 27,000 of 30,000 frames analysed.


### Outputs
| File | Contents |
|---|---|
| `<name>_hbond_frequency_per_residue_all_replicates.csv` | Per residue, per replicate |
| `<name>_hbond_frequency_mean_per_residue.csv` | Mean ± SD across replicates |
| `<name>_hbond_avg_per_frame_summary.csv` | Mean H-bonds per frame per replicate |
| `<name>_hbond_analysis.pkl` | Full results object, consumed by notebook 02 |
| Four PNG figures | Per-replicate bars, mean bars, heatmap, target-vs-total |

### Environment
Google Colab, trajectories on Google Drive. To run locally, remove the Drive
mount and point the paths at a local directory.

## 1. Setup

In [ ]:
# Install required packages
!pip install -q MDAnalysis seaborn pandas matplotlib numpy

# Check GPU availability
!nvidia-smi

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("\nGoogle Drive mounted at /content/drive/MyDrive/")

In [ ]:
import os
import pickle
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import MDAnalysis as mda
from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis as HBA

warnings.filterwarnings('ignore')

print(f"MDAnalysis version: {mda.__version__}")

---
## 2. Configuration — edit this section only

In [ ]:
# ============================================================
# FILE PATHS
# ============================================================
# PDB topology file (shared by all replicates)
pdb_file = 'INSERT_TOPOLOGY_FILENAME.pdb'

# DCD trajectory files (one per replicate)
dcd_files = [
    'INSERT_TRAJECTORY_FILENAME_R1.dcd',
    'INSERT_TRAJECTORY_FILENAME_R2.dcd',
    'INSERT_TRAJECTORY_FILENAME_R3.dcd',
]

# Output directory (results are written here)
output_dir = 'INSERT_OUTPUT_DIRECTORY/'
# ============================================================

In [ ]:
# ============================================================
# CHAINS AND TARGET RESIDUES
# ============================================================
# Residue numbers on the ACCEPTOR_CHAINS side to resolve individually.
# These are resids as they appear in the PDB topology.
TARGET_RESIDUES = [
    147, 148, 149, 150, 151, 152,
    157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
    177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189,
    190, 191, 192, 193, 194, 195, 196, 197, 198, 199,
    200, 201, 202, 203, 204, 205, 206, 207, 208, 209,
]

DONOR_CHAIN = 'C'             # the single chain on one side of the interface
ACCEPTOR_CHAINS = ['A', 'B']  # the chain(s) carrying TARGET_RESIDUES

# ============================================================
# H-BOND GEOMETRIC CRITERIA
# ============================================================
HBOND_DISTANCE = 3.5   # Angstroms - maximum donor-acceptor distance
HBOND_ANGLE = 150      # degrees   - minimum donor-hydrogen-acceptor angle

# ============================================================
# EQUILIBRATION
# ============================================================
# Frames discarded from the START of every replicate. Expressed as a time so
# the value quoted in the methods is explicit and stays correct if the save
# interval changes.
#
# Coordinates were saved every 100 ps -> FRAME_INTERVAL_NS = 0.1
#   1 us replicate            ->  10,000 frames
#   EQUILIBRATION_NS = 100.0  ->   1,000 frames discarded
#
# The first 100 ns of each replicate is discarded, so 1,000 of the 10,000
# frames per replicate are excluded and 27,000 of 30,000 are analyzed.
FRAME_INTERVAL_NS = 0.1
EQUILIBRATION_NS = 100.0

FRAMES_TO_SKIP = int(round(EQUILIBRATION_NS / FRAME_INTERVAL_NS))

# Reference structure for the Ca RMSD:
#   0              -> average over the FULL trajectory (equilibration included)
#   FRAMES_TO_SKIP -> average over the production portion only
RMSD_AVERAGE_START = 0

print(f"Analyzing H-bonds between chain {DONOR_CHAIN} and chains {ACCEPTOR_CHAINS}")
print(f"Target residues: {len(TARGET_RESIDUES)} "
      f"({min(TARGET_RESIDUES)}-{max(TARGET_RESIDUES)})")
print(f"Criteria: distance <= {HBOND_DISTANCE} A, angle >= {HBOND_ANGLE} deg")
print(f"Equilibration discarded: {FRAMES_TO_SKIP} frames "
      f"({EQUILIBRATION_NS} ns) per replicate")

## 3. Verify inputs and create the output directory

In [ ]:
print("Checking files...")
print(f"\nPDB file: {pdb_file}")
print(f"  Exists: {os.path.exists(pdb_file)}")

print(f"\nDCD files:")
for i, dcd in enumerate(dcd_files, 1):
    exists = os.path.exists(dcd)
    size = os.path.getsize(dcd) / (1024**3) if exists else 0
    print(f"  Rep {i}: {dcd}")
    print(f"    Exists: {exists}, Size: {size:.2f} GB")

os.makedirs(output_dir, exist_ok=True)
print(f"\nOutput directory: {output_dir}")
print(f"  Created/exists: {os.path.exists(output_dir)}")

---
## 4. Atom selections and helper functions

Donor, hydrogen and acceptor atoms are given as explicit name lists rather than
left to automatic guessing, so the definitions are identical for every
replicate and for both search directions.

Chains A/B and chain C each contain donors and acceptors, so both chemical
directions are searched and the two results are combined.

In [ ]:
# Explicit atom-name lists. The same lists are applied to whichever side is
# playing the donor or acceptor role, so the two directions are symmetric.
DONOR_NAMES = "name N NE NE1 NE2 ND1 ND2 NH1 NH2 NZ OG OG1 OH"

HYDROGEN_NAMES = ("name H HN HE HE1 HE2 HE21 HE22 HD1 HD2 HD21 HD22 "
                  "HH HH11 HH12 HH21 HH22 HZ HZ1 HZ2 HZ3 HG HG1")

ACCEPTOR_NAMES = "name O OD1 OD2 OE1 OE2 OG OG1 OH ND1 NE2"

In [ ]:
def run_hbond_search(universe, donating_sel, accepting_sel):
    # One directional search: `donating_sel` donates to `accepting_sel`.
    # Calling this twice with the arguments swapped covers both directions.
    hbonds = HBA(
        universe=universe,
        donors_sel=f"({donating_sel}) and ({DONOR_NAMES})",
        hydrogens_sel=f"({donating_sel}) and ({HYDROGEN_NAMES})",
        acceptors_sel=f"({accepting_sel}) and ({ACCEPTOR_NAMES})",
        d_a_cutoff=HBOND_DISTANCE,
        d_h_a_angle_cutoff=HBOND_ANGLE
    )
    hbonds.run(verbose=True)
    return hbonds

In [ ]:
def calculate_rmsd_to_average(universe, selection="protein and name CA", start=0):
    # Per-frame Ca RMSD to the trajectory-average structure.
    #
    # Average positions are accumulated over the trajectory, then each frame is
    # centred and optimally rotated onto the (centred) average with a
    # Kabsch/SVD superposition before the RMSD is evaluated.
    #
    # `start` sets the first frame contributing to the AVERAGE only. An RMSD is
    # returned for every frame regardless, so indices stay absolute.
    sel = universe.select_atoms(selection)
    n_frames = len(universe.trajectory)

    # ---- Pass 1: accumulate the average positions ----------------------
    avg_pos = np.zeros((len(sel), 3))
    n_avg = 0
    for ts in universe.trajectory[start:]:
        avg_pos += sel.positions
        n_avg += 1
    avg_pos /= n_avg

    # ---- Pass 2: superimpose each frame onto the average ---------------
    rmsd_to_avg = np.zeros(n_frames)
    for i, ts in enumerate(universe.trajectory):
        positions = sel.positions - sel.positions.mean(axis=0)
        avg_centered = avg_pos - avg_pos.mean(axis=0)

        # Kabsch rotation from the SVD of the cross-correlation matrix.
        correlation_matrix = np.dot(positions.T, avg_centered)
        U, S, Vt = np.linalg.svd(correlation_matrix)
        rotation = np.dot(U, Vt)

        # Guard against an improper rotation (reflection).
        if np.linalg.det(rotation) < 0:
            Vt[-1, :] *= -1
            rotation = np.dot(U, Vt)

        aligned = np.dot(positions, rotation)
        rmsd_to_avg[i] = np.sqrt(np.mean(np.sum((aligned - avg_centered) ** 2, axis=1)))

    return rmsd_to_avg

### Per-replicate analysis

Four searches are run per replicate — two scopes (target residues, whole
interface) × two directions. Counts are **atom-level**: a residue pair forming
two hydrogen bonds in one frame contributes 2. Frequencies are therefore
reported as *bonds per frame* and can exceed 1.

Separately, each frame's residue-level pairs are recorded as a set — a pair
counts once per frame however many atom-level bonds it forms. Those sets are
what notebook 02 scores frames against; they are not used for the frequencies
reported here.

All per-frame arrays cover the **full** trajectory with absolute frame indices.
The equilibration skip is applied when statistics are computed, never when data
is recorded, so a frame number means the same thing everywhere.

In [ ]:
def analyze_hbonds_replicate(pdb_file, dcd_file, target_residues, replicate_num):
    print(f"\n{'='*60}")
    print(f"Analyzing Replicate {replicate_num}: {os.path.basename(dcd_file)}")
    print('='*60)

    u = mda.Universe(pdb_file, dcd_file)
    print("Guessing bonds from topology...")
    u.atoms.guess_bonds()
    print("Bonds guessed successfully")

    n_frames = len(u.trajectory)
    print(f"Loaded {len(u.atoms)} atoms, {n_frames} frames")
    print(f"  {FRAMES_TO_SKIP} skipped as equilibration, "
          f"{n_frames - FRAMES_TO_SKIP} analyzed")

    if n_frames <= FRAMES_TO_SKIP:
        raise ValueError(
            f"Replicate {replicate_num} has {n_frames} frames but FRAMES_TO_SKIP "
            f"is {FRAMES_TO_SKIP} - nothing would be left to analyze."
        )

    segids = set(u.atoms.segids)
    print(f"Segment IDs in system: {segids}")

    resid_str = ' '.join(map(str, target_residues))
    acc_expr = ' or '.join(f"segid {c}" for c in ACCEPTOR_CHAINS)

    # Prefer segid; fall back to chainID if the topology carries no segids.
    sel_AB_target = f"({acc_expr}) and resid {resid_str}"
    sel_AB_all = f"({acc_expr})"
    sel_C = f"segid {DONOR_CHAIN}"

    atoms_AB_target = u.select_atoms(sel_AB_target)
    atoms_C = u.select_atoms(sel_C)

    if len(atoms_AB_target) == 0 or len(atoms_C) == 0:
        acc_expr = ' or '.join(f"chainID {c}" for c in ACCEPTOR_CHAINS)
        sel_AB_target = f"({acc_expr}) and resid {resid_str}"
        sel_AB_all = f"({acc_expr})"
        sel_C = f"chainID {DONOR_CHAIN}"
        atoms_AB_target = u.select_atoms(sel_AB_target)
        atoms_C = u.select_atoms(sel_C)

    atoms_AB_all = u.select_atoms(sel_AB_all)

    print(f"Chain {'/'.join(ACCEPTOR_CHAINS)} target residues: "
          f"{len(atoms_AB_target)} atoms")
    print(f"Chain {'/'.join(ACCEPTOR_CHAINS)} total: {len(atoms_AB_all)} atoms")
    print(f"Chain {DONOR_CHAIN}: {len(atoms_C)} atoms")

    if len(atoms_AB_target) == 0 or len(atoms_C) == 0:
        raise ValueError("Empty selection - check chain identifiers and resids.")

    chain_attr = 'segid' if sel_C.startswith('segid') else 'chainID'

    # ================================================================
    # ANALYSIS 1: target residues  (C <-> specific residues in A/B)
    # ================================================================
    print("\n--- Analyzing TARGET RESIDUE H-bonds ---")
    print(f"Running H-bond analysis ({DONOR_CHAIN} -> target residues)...")
    hbonds_target_1 = run_hbond_search(u, sel_C, sel_AB_target)
    print(f"Running H-bond analysis (target residues -> {DONOR_CHAIN})...")
    hbonds_target_2 = run_hbond_search(u, sel_AB_target, sel_C)

    results_target_1 = hbonds_target_1.results.hbonds
    results_target_2 = hbonds_target_2.results.hbonds
    print(f"H-bonds found ({DONOR_CHAIN} -> target): {len(results_target_1)}")
    print(f"H-bonds found (target -> {DONOR_CHAIN}): {len(results_target_2)}")

    # ================================================================
    # ANALYSIS 2: whole interface  (C <-> ALL of A/B)
    # ================================================================
    print("\n--- Analyzing TOTAL interface H-bonds ---")
    print(f"Running H-bond analysis ({DONOR_CHAIN} -> all A/B)...")
    hbonds_total_1 = run_hbond_search(u, sel_C, sel_AB_all)
    print(f"Running H-bond analysis (all A/B -> {DONOR_CHAIN})...")
    hbonds_total_2 = run_hbond_search(u, sel_AB_all, sel_C)

    results_total_1 = hbonds_total_1.results.hbonds
    results_total_2 = hbonds_total_2.results.hbonds
    print(f"H-bonds found ({DONOR_CHAIN} -> all A/B): {len(results_total_1)}")
    print(f"H-bonds found (all A/B -> {DONOR_CHAIN}): {len(results_total_2)}")

    # ================================================================
    # COUNT TARGET-RESIDUE H-BONDS (production frames only)
    # ================================================================
    # results.hbonds columns:
    #   [frame, donor_idx, hydrogen_idx, acceptor_idx, distance, angle]
    per_residue_counts = defaultdict(int)
    per_frame_counts_target = np.zeros(n_frames, dtype=int)
    frame_pairs = [set() for _ in range(n_frames)]

    def tally_target(results, ab_col):
        # ab_col: column holding the A/B-side atom (1 = donor, 3 = acceptor).
        c_col = 3 if ab_col == 1 else 1
        for hbond in results:
            frame = int(hbond[0])
            ab_atom = u.atoms[int(hbond[ab_col])]
            c_atom = u.atoms[int(hbond[c_col])]

            chain = getattr(ab_atom, chain_attr)
            resid = ab_atom.resid
            if resid not in target_residues:
                continue

            # Residue-level pair, keyed (A/B residue, C residue) in BOTH
            # directions so the two searches describe the same pair rather than
            # two mirror-image entries. Consumed only by notebook 02.
            pair = (f"{ab_atom.resname}{resid}:{chain}",
                    f"{c_atom.resname}{c_atom.resid}:{getattr(c_atom, chain_attr)}")
            frame_pairs[frame].add(pair)

            if frame < FRAMES_TO_SKIP:
                continue
            per_residue_counts[f"{chain}{resid}"] += 1
            per_frame_counts_target[frame] += 1

    tally_target(results_target_1, ab_col=3)  # C donated   -> A/B accepted
    tally_target(results_target_2, ab_col=1)  # A/B donated -> C accepted

    # ================================================================
    # COUNT TOTAL INTERFACE H-BONDS (production frames only)
    # ================================================================
    per_frame_counts_total = np.zeros(n_frames, dtype=int)
    for results in (results_total_1, results_total_2):
        for hbond in results:
            per_frame_counts_total[int(hbond[0])] += 1

    # ================================================================
    # PER-FRAME Ca RMSD TO THE REPLICATE AVERAGE
    # ================================================================
    print("\nCalculating Ca RMSD to the average structure...")
    rmsd_values = calculate_rmsd_to_average(u, selection="protein and name CA",
                                            start=RMSD_AVERAGE_START)

    # ================================================================
    # STATISTICS over production frames
    # ================================================================
    analyzed_frames = n_frames - FRAMES_TO_SKIP

    total_hbonds_target = int(per_frame_counts_target[FRAMES_TO_SKIP:].sum())
    avg_hbonds_target = total_hbonds_target / analyzed_frames

    total_hbonds_all = int(per_frame_counts_total[FRAMES_TO_SKIP:].sum())
    avg_hbonds_total = total_hbonds_all / analyzed_frames

    print(f"\n=== SUMMARY ===")
    print(f"Target residue H-bonds: {total_hbonds_target} total, "
          f"{avg_hbonds_target:.2f} avg/frame")
    print(f"Total interface H-bonds: {total_hbonds_all} total, "
          f"{avg_hbonds_total:.2f} avg/frame")

    replicate_record = {
        'name': f"Rep{replicate_num}",
        'n_frames': n_frames,
        'analyzed_frames': analyzed_frames,
        'hbond_counts': per_frame_counts_target,
        'hbond_counts_total': per_frame_counts_total,
        'rmsd_values': rmsd_values,
        'frame_pairs': frame_pairs,
    }

    return (dict(per_residue_counts), avg_hbonds_target, avg_hbonds_total,
            n_frames, analyzed_frames, replicate_record)

---
## 5. Run the analysis

> Runtime note: each replicate is read six times (four H-bond searches plus the
> two-pass RMSD), and `guess_bonds()` on a solvated system is itself slow. This
> is by far the longest cell.

In [ ]:
all_results = []
avg_hbonds_summary = []
replicate_records = []

for i, dcd_file in enumerate(dcd_files, 1):
    (per_res, avg_hb_target, avg_hb_total,
     n_frames, analyzed_frames, rec) = analyze_hbonds_replicate(
        pdb_file, dcd_file, TARGET_RESIDUES, i
    )

    all_results.append(per_res)
    replicate_records.append(rec)
    avg_hbonds_summary.append({
        'Replicate': f'Rep {i}',
        'Trajectory': os.path.basename(dcd_file),
        'Frames': n_frames,
        'Frames_Analyzed': analyzed_frames,
        'Avg H-bonds/frame (target)': avg_hb_target,
        'Avg H-bonds/frame (total C-AB)': avg_hb_total,
    })

print("\n" + "="*60)
print("Analysis Complete!")
print("="*60)

### Optional diagnostic — hydrogen names present in the topology

In [ ]:
# Confirms the HYDROGEN_NAMES list actually matches this force field's naming.
u_check = mda.Universe(pdb_file, dcd_files[0])
acc_expr = ' or '.join(f"segid {c}" for c in ACCEPTOR_CHAINS)
ab_atoms = u_check.select_atoms(f"({acc_expr})")
if len(ab_atoms) == 0:
    acc_expr = ' or '.join(f"chainID {c}" for c in ACCEPTOR_CHAINS)
    ab_atoms = u_check.select_atoms(f"({acc_expr})")

present = sorted(set(ab_atoms.select_atoms("name H*").names))
print("Hydrogen names in A/B:", present)

listed = set(HYDROGEN_NAMES.replace('name ', '').split())
print("\nPresent in topology but NOT in HYDROGEN_NAMES:",
      sorted(n for n in present if n not in listed))

---
## 6. Summary tables

In [ ]:
summary_df = pd.DataFrame(avg_hbonds_summary)
print("\n=== Average H-bonds per Frame Summary ===")
print(f"(production frames only: first {FRAMES_TO_SKIP} of each replicate excluded)")
print(summary_df.to_string(index=False))

print(f"\nTarget residues - Overall mean: "
      f"{summary_df['Avg H-bonds/frame (target)'].mean():.2f} ± "
      f"{summary_df['Avg H-bonds/frame (target)'].std():.2f}")
print(f"Total C <-> A/B - Overall mean: "
      f"{summary_df['Avg H-bonds/frame (total C-AB)'].mean():.2f} ± "
      f"{summary_df['Avg H-bonds/frame (total C-AB)'].std():.2f}")

In [ ]:
# Per-residue frequency table (bonds per production frame)
residue_labels = []
for chain in ACCEPTOR_CHAINS:
    for resid in TARGET_RESIDUES:
        residue_labels.append(f"{chain}{resid}")

plot_data = []
for i, res_counts in enumerate(all_results, 1):
    analyzed = avg_hbonds_summary[i-1]['Frames_Analyzed']
    for label in residue_labels:
        count = res_counts.get(label, 0)
        plot_data.append({
            'Residue': label,
            'Replicate': f'Rep {i}',
            'Count': count,
            'Frequency': count / analyzed if analyzed > 0 else 0,
        })

df = pd.DataFrame(plot_data)
print(df.head(50))

In [ ]:
mean_df = df.groupby('Residue').agg(
    Mean_Frequency=('Frequency', 'mean'),
    Std_Frequency=('Frequency', 'std'),
    Total_Count=('Count', 'sum')
).reset_index()

mean_df['Residue'] = pd.Categorical(mean_df['Residue'],
                                    categories=residue_labels, ordered=True)
mean_df = mean_df.sort_values('Residue')
print(mean_df.to_string(index=False))

## 7. Figures

In [ ]:
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
palette = sns.color_palette("husl", n_colors=len(dcd_files))

structure_name = os.path.basename(pdb_file).replace('.pdb', '').replace('.PDB', '')
chain_pair_label = f"Chain {DONOR_CHAIN} ↔ Chain {'/'.join(ACCEPTOR_CHAINS)}"

In [ ]:
# PLOT 1: per-residue frequency, all replicates
fig, ax = plt.subplots(figsize=(16, 8))

sns.barplot(
    data=df, x='Residue', y='Frequency', hue='Replicate',
    palette=palette, ax=ax, edgecolor='black', linewidth=0.5
)

ax.set_xlabel('Residue', fontsize=14, fontweight='bold')
ax.set_ylabel('H-bond Frequency (bonds/frame)', fontsize=14, fontweight='bold')
ax.set_title(f'Hydrogen Bond Frequency: {chain_pair_label} Residues',
             fontsize=16, fontweight='bold', pad=20)

plt.xticks(rotation=45, ha='right', fontsize=11)
ax.legend(title='Replicate', title_fontsize=12, fontsize=11, loc='upper right')

# Separator between the two acceptor chains
ax.axvline(x=len(TARGET_RESIDUES) - 0.5, color='gray', linestyle='--',
           linewidth=2, alpha=0.7)
ax.text(len(TARGET_RESIDUES)/2 - 0.5, ax.get_ylim()[1]*0.95,
        f'Chain {ACCEPTOR_CHAINS[0]}', ha='center', fontsize=12,
        fontweight='bold', color='darkblue')
ax.text(len(TARGET_RESIDUES) + len(TARGET_RESIDUES)/2 - 0.5,
        ax.get_ylim()[1]*0.95, f'Chain {ACCEPTOR_CHAINS[1]}', ha='center',
        fontsize=12, fontweight='bold', color='darkred')

plt.tight_layout()
plt.savefig(f'{output_dir}{structure_name}_hbond_frequency_by_replicate.png',
            bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# PLOT 2: mean frequency with error bars
fig, ax = plt.subplots(figsize=(16, 8))

colors = ['#3498db' if res.startswith(ACCEPTOR_CHAINS[0]) else '#e74c3c'
          for res in mean_df['Residue']]

bars = ax.bar(
    range(len(mean_df)), mean_df['Mean_Frequency'],
    yerr=mean_df['Std_Frequency'], capsize=4, color=colors,
    edgecolor='black', linewidth=1,
    error_kw={'linewidth': 1.5, 'capthick': 1.5}
)

ax.set_xticks(range(len(mean_df)))
ax.set_xticklabels(mean_df['Residue'], rotation=45, ha='right', fontsize=12)
ax.set_xlabel('Residue', fontsize=14, fontweight='bold')
ax.set_ylabel('Mean H-bond Frequency (bonds/frame)', fontsize=14, fontweight='bold')
ax.set_title(f'Mean Hydrogen Bond Frequency Across Replicates\n({chain_pair_label})',
             fontsize=16, fontweight='bold')

ax.axvline(x=len(TARGET_RESIDUES) - 0.5, color='gray', linestyle='--',
           linewidth=2, alpha=0.7)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#3498db', edgecolor='black',
          label=f'Chain {ACCEPTOR_CHAINS[0]}'),
    Patch(facecolor='#e74c3c', edgecolor='black',
          label=f'Chain {ACCEPTOR_CHAINS[1]}')
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=11)

plt.tight_layout()
plt.savefig(f'{output_dir}{structure_name}_hbond_frequency_mean.png',
            bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# PLOT 3: heatmap
heatmap_data = df.pivot(index='Replicate', columns='Residue', values='Frequency')
heatmap_data = heatmap_data[residue_labels]

fig, ax = plt.subplots(figsize=(16, 5))

sns.heatmap(
    heatmap_data, annot=True, fmt='.3f', cmap='YlOrRd', linewidths=0.5, ax=ax,
    cbar_kws={'label': 'H-bond Frequency (bonds/frame)', 'shrink': 0.8}
)

ax.set_xlabel('Residue', fontsize=14, fontweight='bold')
ax.set_ylabel('Replicate', fontsize=14, fontweight='bold')
ax.set_title(f'H-bond Frequency Heatmap: {chain_pair_label} Residues',
             fontsize=16, fontweight='bold', pad=15)

plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(f'{output_dir}{structure_name}_hbond_frequency_heatmap.png',
            bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# PLOT 4: target residues vs whole interface
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, col, cmap, title in (
    (axes[0], 'Avg H-bonds/frame (target)', 'Blues_d',
     f'Target Residue H-bonds\n({chain_pair_label} specific residues)'),
    (axes[1], 'Avg H-bonds/frame (total C-AB)', 'Greens_d',
     f'Total Interface H-bonds\n({chain_pair_label}, all residues)'),
):
    sns.barplot(data=summary_df, x='Replicate', y=col, palette=cmap, ax=ax,
                edgecolor='black', linewidth=1.5)

    for i, bar in enumerate(ax.patches):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{summary_df.iloc[i][col]:.2f}', ha='center',
                fontsize=12, fontweight='bold')

    mean_val = summary_df[col].mean()
    ax.axhline(y=mean_val, color='red', linestyle='--', linewidth=2,
               label=f'Mean: {mean_val:.2f}')
    ax.set_xlabel('Replicate', fontsize=14, fontweight='bold')
    ax.set_ylabel('Avg H-bonds per Frame', fontsize=14, fontweight='bold')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.legend(loc='upper right', fontsize=11)

plt.tight_layout()
plt.savefig(f'{output_dir}{structure_name}_hbond_comparison_target_vs_total.png',
            bbox_inches='tight', facecolor='white')
plt.show()

## 8. Final summary

In [ ]:
print("\n" + "="*80)
print("FINAL RESULTS SUMMARY")
print("="*80)
print(f"Criteria: <= {HBOND_DISTANCE} A donor-acceptor, >= {HBOND_ANGLE} deg")
print(f"Equilibration excluded: first {FRAMES_TO_SKIP} frames "
      f"({EQUILIBRATION_NS} ns) of each replicate")

print("\n1. AVERAGE H-BONDS PER FRAME FOR EACH REPLICATE:")
print("-"*50)
for i, row in summary_df.iterrows():
    print(f"   {row['Replicate']}:")
    print(f"      Target residues: {row['Avg H-bonds/frame (target)']:.4f} bonds/frame")
    print(f"      Total C <-> A/B: {row['Avg H-bonds/frame (total C-AB)']:.4f} bonds/frame")
    print(f"      ({row['Frames_Analyzed']} of {row['Frames']} frames analyzed)")

print(f"\n   Target residues - Overall Mean: "
      f"{summary_df['Avg H-bonds/frame (target)'].mean():.4f} ± "
      f"{summary_df['Avg H-bonds/frame (target)'].std():.4f}")
print(f"   Total C <-> A/B - Overall Mean: "
      f"{summary_df['Avg H-bonds/frame (total C-AB)'].mean():.4f} ± "
      f"{summary_df['Avg H-bonds/frame (total C-AB)'].std():.4f}")

print("\n2. PER-RESIDUE MEAN H-BOND FREQUENCY:")
print("-"*50)
print(mean_df.to_string(index=False))

print("\n3. TOP 5 RESIDUES BY H-BOND FREQUENCY:")
print("-"*50)
for _, row in mean_df.nlargest(5, 'Mean_Frequency').iterrows():
    print(f"   {row['Residue']}: {row['Mean_Frequency']:.4f} ± {row['Std_Frequency']:.4f}")

---
## 9. Export CSVs and the results object

The pickle holds the per-frame residue-pair sets and Cα RMSD traces that
`02_representative_frame_analysis.ipynb` needs.

In [ ]:
df.to_csv(f'{output_dir}{structure_name}_hbond_frequency_per_residue_all_replicates.csv',
          index=False)
mean_df.to_csv(f'{output_dir}{structure_name}_hbond_frequency_mean_per_residue.csv',
               index=False)
summary_df.to_csv(f'{output_dir}{structure_name}_hbond_avg_per_frame_summary.csv',
                  index=False)

print(f"\nResults saved to: {output_dir}")
for suffix in ('hbond_frequency_per_residue_all_replicates.csv',
               'hbond_frequency_mean_per_residue.csv',
               'hbond_avg_per_frame_summary.csv',
               'hbond_frequency_by_replicate.png',
               'hbond_frequency_mean.png',
               'hbond_frequency_heatmap.png',
               'hbond_comparison_target_vs_total.png'):
    print(f"  - {structure_name}_{suffix}")

In [ ]:
# Results object consumed by notebook 02.
all_results_pkl = {
    structure_name: {
        'config': {
            'pdb_file': pdb_file,
            'dcd_files': dcd_files,
            'target_residues': TARGET_RESIDUES,
            'donor_chain': DONOR_CHAIN,
            'acceptor_chains': ACCEPTOR_CHAINS,
            'hbond_distance': HBOND_DISTANCE,
            'hbond_angle': HBOND_ANGLE,
        },
        'replicates': replicate_records,
        'frames_to_skip': FRAMES_TO_SKIP,
        'equilibration_ns': EQUILIBRATION_NS,
        'directions': 'both',
    }
}

pkl_path = f'{output_dir}{structure_name}_hbond_analysis.pkl'
with open(pkl_path, 'wb') as f:
    pickle.dump(all_results_pkl, f)

print(f"Saved results object: {pkl_path}")